## Read the data

In [50]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"
EXPERIMENT= "core_identification_paper"

# 1. Read data (Now it is the same data)

In [51]:
DATA_PATH_BASE = 'G:/My Drive/Work/ITMO/data'
# DATA_PATH_BASE = 'D:/Google Drive/Work/ITMO/data' #windows
# DATA_PATH_BASE = '/home/denklewer/Data' #ubuntu

DATA_PATH = f'{DATA_PATH_BASE}/{EXPERIMENT}/per_length_per_kfold_split/' # work laptop
PATH_TO_SAVE_RESULTS = f'{DATA_PATH_BASE}/{EXPERIMENT}/peptide_cores/'
APPROACH = "DECOY_PATHS"

In [52]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = list(sorted(get_available_alleles(DATA_PATH)))

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
# per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
# per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

target_class = "class2"


DataParams = {
"class1": {
    "ALLELES": sorted([ 'HLA-A*01:01', 'HLA-A*03:01']),
    "TARGET_LENGTHS":   [8, 9, 10, 11, 12],
},
    
"class2": {
    "ALLELES":sorted(['HLA-DPA1*02:02/DPB1*05:01']),
  #  "ALLELES":sorted(['HLA-DRB1*01:01']),
    "TARGET_LENGTHS":    [12, 13,14, 15, 16,17, 18],

    
    
}

}

ALLELES = DataParams[target_class]["ALLELES"]
TARGET_LENGTHS = DataParams[target_class]["TARGET_LENGTHS"]

selected_alleles = ALLELES
ALLELES




['HLA-DPA1*02:02/DPB1*05:01']

## 2.0 Merge alleles together

In [53]:
# combination_size = len(ALLELES)
# ALLELES = sorted(['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'])
# # ALLELES = sorted([ 'HLA-A*03:01'])
# ALLELES = sorted([   'HLA-DRB1*01:01',
#                     # 'HLA-DPA1*01:03/DPB1*04:01'
#                       ])
# ALLELES = sorted(['HLA-DPA1*01:03-DPB1*04:01'])
#
# ALLELES = sorted(['HLA-DRB1*01:01', 'HLA-DPA1*01:03-DPB1*04:01'])
#

combination_size = 2
TOTAL_SPLITS = 1
# TARGET_LENGTHS= [8, 9, 10, 11, 12]
RUNS = 25
USE_WEIGHTS = True

# SELECTED_RUN = 0
# SELECTED_RUN = {0,1,2,3,4,5, 6, 7, 8, 9}
SELECTED_RUN = {0,1}

# SELECTED_RUN = {0}
#
SELECTED_SPLIT = 0
models_to_read = ['v']

## Merge train data together (multiple alleles to single set of peptides)

In [54]:
from collections import defaultdict
global_selected_alleles = ALLELES # Data alleles we selected (but the number will be reduced by acceptable alleles)

selected_alleles = global_selected_alleles


per_allele_representations_train = dict()
per_allele_representations_test = dict()
per_name_representation_train = dict()
per_name_representation_test = dict()

def handle_intersections(per_allele_per_kfold_per_length_peptides):
    per_allele_result_peptide_sets =  defaultdict(set)
    per_allele_per_length_peptide_sets =  defaultdict(set)
    PER_LENGTH_SIZE = {target_length: min([len(set(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length])) for allele_name in selected_alleles]) for target_length in TARGET_LENGTHS}
    print("Min unique sizes", PER_LENGTH_SIZE)
    for target_length in TARGET_LENGTHS:
        print(f"Length {target_length}")
        for allele_name in selected_alleles:
            if target_length in per_allele_per_kfold_per_length_peptides[allele_name][0]:
                print(f"initial length for  allele { allele_name} TL {target_length} is {len(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length])}")
                print(f"unique length for  allele { allele_name} TL {target_length} is {len(set(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length]))}")
                unique_peptides = np.unique(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length])
                if "imm" not in allele_name: #do not subsambple for imm
                    peptides = set(np.random.choice(unique_peptides, size=PER_LENGTH_SIZE[target_length], replace=False))
                else:
                    peptides = set(unique_peptides)
                print(f"Selected/Subsampled {len(peptides)} peptides")
                per_allele_per_length_peptide_sets[allele_name] = peptides
        # get all intersections
        intersections = list()
        for i in range(len(selected_alleles)):
            for j in range(i + 1, len(selected_alleles)):
                set1 = per_allele_per_length_peptide_sets[selected_alleles[i]]
                set2 = per_allele_per_length_peptide_sets[selected_alleles[j]]
                intersect = set1.intersection(set2)
                print(f"{selected_alleles[i]} intersect {selected_alleles[j]} - {len(intersect)}")
                intersections.append(intersect)
        # delete all intersections
        for allele_name, peptide_set in per_allele_per_length_peptide_sets.items():
            print(f"Allele {allele_name}: deleted", end=" ")
            initial_length = len(peptide_set)
            for intersect in intersections:
                peptide_set = peptide_set.difference(intersect)
            result_length = len(peptide_set)
            print(f" {initial_length - result_length} of intersected peptides")
            per_allele_result_peptide_sets[allele_name] = per_allele_result_peptide_sets[allele_name].union(peptide_set)
    return per_allele_result_peptide_sets

print("Train")
per_allele_peptide_train_sets = handle_intersections(per_allele_per_kfold_per_length_binders_train)
print("Test")
per_allele_peptide_test_sets = handle_intersections(per_allele_per_kfold_per_length_binders_test)



train_peptides_b = set(set().union(*per_allele_peptide_train_sets.values()))

test_peptides_b = set(set().union(*per_allele_peptide_test_sets.values()))

print(f"Train {len(train_peptides_b)}, Test {len(test_peptides_b)}")



data_df = pd.DataFrame(index=list(train_peptides_b))
for allele_name in selected_alleles:
    data_df.loc[list(per_allele_peptide_train_sets[allele_name]), 'allele'] = allele_name



Train
Min unique sizes {12: 420, 13: 790, 14: 1241, 15: 1231, 16: 927, 17: 628, 18: 403}
Length 12
initial length for  allele HLA-DPA1*02:02/DPB1*05:01 TL 12 is 420
unique length for  allele HLA-DPA1*02:02/DPB1*05:01 TL 12 is 420
Selected/Subsampled 420 peptides
Allele HLA-DPA1*02:02/DPB1*05:01: deleted  0 of intersected peptides
Length 13
initial length for  allele HLA-DPA1*02:02/DPB1*05:01 TL 13 is 790
unique length for  allele HLA-DPA1*02:02/DPB1*05:01 TL 13 is 790
Selected/Subsampled 790 peptides
Allele HLA-DPA1*02:02/DPB1*05:01: deleted  0 of intersected peptides
Length 14
initial length for  allele HLA-DPA1*02:02/DPB1*05:01 TL 14 is 1241
unique length for  allele HLA-DPA1*02:02/DPB1*05:01 TL 14 is 1241
Selected/Subsampled 1241 peptides
Allele HLA-DPA1*02:02/DPB1*05:01: deleted  0 of intersected peptides
Length 15
initial length for  allele HLA-DPA1*02:02/DPB1*05:01 TL 15 is 1231
unique length for  allele HLA-DPA1*02:02/DPB1*05:01 TL 15 is 1231
Selected/Subsampled 1231 peptides
Al

In [42]:
from data_reading_methods import merge_per_allele_sets_into_combination_sets

per_allele_per_kfold_per_length_binders_train_old = per_allele_per_kfold_per_length_binders_train
per_allele_per_kfold_per_length_binders_test_old = per_allele_per_kfold_per_length_binders_test

per_allele_per_kfold_per_length_binders_train, per_allele_per_kfold_per_length_binders_test = merge_per_allele_sets_into_combination_sets(
    per_allele_per_kfold_per_length_binders_train, per_allele_per_kfold_per_length_binders_test,
    combination_size=combination_size,
    selected_alelles=ALLELES,
    total_splits=TOTAL_SPLITS
)
OLD_ALLELES = ALLELES
NEW_ALLELES = list(per_allele_per_kfold_per_length_binders_test.keys())
ALLELES = NEW_ALLELES

In [55]:
ALLELES

['HLA-DPA1*02:02/DPB1*05:01']

In [46]:
per_allele_peptide_train_sets.keys()

dict_keys(['HLA-DPA1*02:02/DPB1*05:01'])

In [56]:
selected_alleles

['HLA-DPA1*02:02/DPB1*05:01']

In [57]:
data_df.allele.value_counts()

HLA-DPA1*02:02/DPB1*05:01    5640
Name: allele, dtype: int64

# Read models

In [58]:
MODELS_DESCRIPTIONS = {
                    "core_identification_paper_class2_chains": {
        "path": "experiment_results/batched_and_chains/",
        "states": 14,
        "params_string": "1000-min_iters",
        "combination_size": 1,
        "alleles": [
                # 'HLA-DRB1*01:01'
                    'HLA-DPA1*02:02/DPB1*05:01',
                    #'HLA-A*03:01'
                    ],
        "target_lengths": [12, 13,14, 15, 16, 17, 18],
        "core_start_state": [14, 15],
        "core_end_state": [16,17],
        "need_to_delete": [111],
        "skip_states": [],

    },

}

In [59]:
selected_models = [
     "core_identification_paper_class2_chains"
]


In [60]:
from hmm_logic_methods import load_model

In [61]:
import itertools

name_to_run = dict()
name_to_split = dict()
name_to_type = dict()
name_to_model_root = dict()
name_to_target_lengths = dict()
selected_binding_models = list()
per_name_binding_model = dict()


def populate_models_infos(model_name, run_num, split_num, model_path, model_type, target_lengths ):
    selected_binding_models.append(model_name)

    name_to_run[model_name] = run_num
    name_to_split[model_name] = split_num
    name_to_model_root[model_name] = model_path
    assert model_type in ["bw", "v", "bwv", "bw-it", "bwv-it", "v-it"] or "mixed" in model_type
    name_to_type[model_name] = model_type
    name_to_target_lengths[model_name] = target_lengths

for model in selected_models:
    model_description = MODELS_DESCRIPTIONS[model]
    combination_size = model_description['combination_size']
    model_alleles = model_description['alleles']
    MODELS_PATH = model_description['path']
    params_string = model_description['params_string']
    states = model_description['states']
    TARGET_LENGTHS = model_description['target_lengths']
    target_length_string = "-".join([str(l) for l in TARGET_LENGTHS])
    MIXED_ALLELES = list()

    model_alleles = set(model_alleles).intersection(set(OLD_ALLELES))
    for combination in itertools.combinations(sorted(model_alleles), combination_size):
        allele_name = "+".join(combination)
        # allele_name = f"{allele_name1}+{allele_name2}"
        MIXED_ALLELES.append(allele_name)

    for allele_name in MIXED_ALLELES:
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_').replace('/', '_')
        for run_num_b in range(RUNS):
            for split_num_b in range(TOTAL_SPLITS):
                #bw
                if 'bw' in models_to_read:
                    bw_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    print(bw_model_name)
                    populate_models_infos(bw_model_name, run_num_b,split_num_b, MODELS_PATH, f"bw-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'bwv' in models_to_read:
                    #bwv
                    bwv_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}_plus_viterbi"
                    populate_models_infos(bwv_model_name, run_num_b,split_num_b, MODELS_PATH, f"bwv-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'v' in models_to_read:
                    #v
                    v_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-viterbi-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    populate_models_infos(v_model_name, run_num_b,split_num_b, MODELS_PATH, f"v-it-mixed({target_allele_name})", TARGET_LENGTHS)



for model_name in selected_binding_models:
    per_name_binding_model[model_name] =  load_model(name_to_model_root[model_name], model_name)

In [62]:
model_alleles = model_description['alleles']
MIXED_ALLELES = list()

model_alleles = set(model_alleles).intersection(set(OLD_ALLELES))
for combination in itertools.combinations(sorted(model_alleles), combination_size):
    allele_name = "+".join(combination)
    # allele_name = f"{allele_name1}+{allele_name2}"
    MIXED_ALLELES.append(allele_name)

In [63]:
model_description['alleles']

['HLA-DPA1*02:02/DPB1*05:01']

In [64]:
OLD_ALLELES

['HLA-DPA1*02:02/DPB1*05:01']

In [65]:
model_alleles

{'HLA-DPA1*02:02/DPB1*05:01'}

In [66]:
MODELS_DESCRIPTIONS[model]

{'path': 'experiment_results/batched_and_chains/',
 'states': 14,
 'params_string': '1000-min_iters',
 'combination_size': 1,
 'alleles': ['HLA-DPA1*02:02/DPB1*05:01'],
 'target_lengths': [12, 13, 14, 15, 16, 17, 18],
 'core_start_state': [14, 15],
 'core_end_state': [16, 17],
 'need_to_delete': [111],
 'skip_states': []}

In [67]:
MODELS_DESCRIPTIONS[model]

{'path': 'experiment_results/batched_and_chains/',
 'states': 14,
 'params_string': '1000-min_iters',
 'combination_size': 1,
 'alleles': ['HLA-DPA1*02:02/DPB1*05:01'],
 'target_lengths': [12, 13, 14, 15, 16, 17, 18],
 'core_start_state': [14, 15],
 'core_end_state': [16, 17],
 'need_to_delete': [111],
 'skip_states': []}

## Mark important states in model(will be used for core call)

In [68]:
import logomaker
from collections import defaultdict
def probs_to_info_matrix(distribution):
    amino_acids = list(distribution.parameters[0].keys())
    dummy_df = pd.DataFrame(columns = sorted(amino_acids), index=[0],  dtype=float)
    for i, (acid, prob) in enumerate(distribution.parameters[0].items()):
        dummy_df.loc[0, acid] = prob
    dummy_df.index.name = "pos"
    info_matrix = logomaker.transform_matrix(dummy_df, from_type='probability', to_type='information')
    return info_matrix

per_model_total_df = dict()

for model_key, model in per_name_binding_model.items():
    per_state_information = dict()
    per_state_distribution = dict()
    per_level_information = defaultdict(float)

    for state in model.states:
        if state not in [model.start, model.end]:
            info_matrix = probs_to_info_matrix(state.distribution)
            per_state_distribution[state.name] = state.distribution.parameters[0]
            per_state_information[state.name] = sum(info_matrix.loc[0])
            # per_level_information[name_to_level[state.name]] += per_state_information[state.name]
    info_df = pd.DataFrame(per_state_information.items(), columns=["state", "info"])
    distr_df = pd.DataFrame(per_state_distribution.values())
    total_df = pd.concat([info_df, distr_df], axis=1)
    per_model_total_df[model_key] = total_df

In [69]:
filtered_dfs = list()
per_model_important_states = dict()
per_model_skip_states = dict()

if models_to_read == ['bw']:
    for i, (model_key, total_df) in enumerate(per_model_total_df.items()):
        total_df = total_df.sort_values(by='info', ascending=False).copy()
        total_df = total_df[total_df['info'] >= 1]
        total_df['model'] = i
        total_df['model_name'] = model_key
        filtered_dfs.append(total_df)
        per_model_important_states[model_key] = list(total_df['state'].values)

if models_to_read == ['v']:
    per_model_anchors = dict()
    per_name_models = per_name_binding_model

    core_start_states= MODELS_DESCRIPTIONS[selected_models[0]]['core_start_state']
    core_end_states = MODELS_DESCRIPTIONS[selected_models[0]]['core_end_state']
    skip_states =  MODELS_DESCRIPTIONS[selected_models[0]]['skip_states']

    for i, (model_key, total_df) in enumerate(per_model_total_df.items()):
        total_df = total_df.sort_values(by='info', ascending=False).copy()
        total_df = total_df.loc[core_start_states + core_end_states]
        total_df['model'] = i
        total_df['model_name'] = model_key
        filtered_dfs.append(total_df)
        per_model_important_states[model_key] = list(total_df['state'].values)
    for i, (model_key, total_df) in enumerate(per_model_total_df.items()):
  
        total_df = total_df.sort_values(by='info', ascending=False).copy()
        total_df = total_df.loc[skip_states]
        total_df['model'] = i
        total_df['model_name'] = model_key
        filtered_dfs.append(total_df)
        per_model_skip_states[model_key] = list(total_df['state'].values)
        





In [70]:
data_df

,allele
MPQVPVFTVEVKNYGVK,HLA-DPA1*02:02/DPB1*05:01
RTELETLQKQKK,HLA-DPA1*02:02/DPB1*05:01
SQEKQVTVLELFR,HLA-DPA1*02:02/DPB1*05:01
IIMDVYSYSEKE,HLA-DPA1*02:02/DPB1*05:01
QALVEHVKEECD,HLA-DPA1*02:02/DPB1*05:01
...,...
GATGAIMTTETTKTARAT,HLA-DPA1*02:02/DPB1*05:01
STKTSAKLSSYT,HLA-DPA1*02:02/DPB1*05:01
IASRLGLYSKTSQS,HLA-DPA1*02:02/DPB1*05:01
ELSAWTEKQKEERQF,HLA-DPA1*02:02/DPB1*05:01


In [71]:
ALLELES

['HLA-DPA1*02:02/DPB1*05:01']

In [72]:
data_df[data_df['allele'].map(lambda x : x in "HLA-A*01:01_123")]

,allele


In [73]:
data_df['allele'].map(lambda x : x in ALLELES[0])

MPQVPVFTVEVKNYGVK     True
RTELETLQKQKK          True
SQEKQVTVLELFR         True
IIMDVYSYSEKE          True
QALVEHVKEECD          True
                      ... 
GATGAIMTTETTKTARAT    True
STKTSAKLSSYT          True
IASRLGLYSKTSQS        True
ELSAWTEKQKEERQF       True
SVEEKRNSKMLQK         True
Name: allele, Length: 5640, dtype: bool

## Calculate core guesses

In [74]:
    parsed_alleles = [mhcgnomes.parse(curr_name).compact_string(include_species=True).replace("*", "_") for curr_name in allele_name.split("+")]


NameError: name 'mhcgnomes' is not defined

In [117]:
parsed_alleles

['HLA-A0101', 'HLA-A0301']

In [36]:
ALLELES

[]

In [35]:
per_name_models.keys()

dict_keys(['0_run-12-13-14-15-16-17-18-14-st-viterbi-alg-1000-min_iters_model-HLA_DPA1_02_02_DPB1_05_01-0', '1_run-12-13-14-15-16-17-18-14-st-viterbi-alg-1000-min_iters_model-HLA_DPA1_02_02_DPB1_05_01-0', '2_run-12-13-14-15-16-17-18-14-st-viterbi-alg-1000-min_iters_model-HLA_DPA1_02_02_DPB1_05_01-0', '3_run-12-13-14-15-16-17-18-14-st-viterbi-alg-1000-min_iters_model-HLA_DPA1_02_02_DPB1_05_01-0', '4_run-12-13-14-15-16-17-18-14-st-viterbi-alg-1000-min_iters_model-HLA_DPA1_02_02_DPB1_05_01-0', '5_run-12-13-14-15-16-17-18-14-st-viterbi-alg-1000-min_iters_model-HLA_DPA1_02_02_DPB1_05_01-0', '6_run-12-13-14-15-16-17-18-14-st-viterbi-alg-1000-min_iters_model-HLA_DPA1_02_02_DPB1_05_01-0', '7_run-12-13-14-15-16-17-18-14-st-viterbi-alg-1000-min_iters_model-HLA_DPA1_02_02_DPB1_05_01-0', '8_run-12-13-14-15-16-17-18-14-st-viterbi-alg-1000-min_iters_model-HLA_DPA1_02_02_DPB1_05_01-0', '9_run-12-13-14-15-16-17-18-14-st-viterbi-alg-1000-min_iters_model-HLA_DPA1_02_02_DPB1_05_01-0', '10_run-12-13-14-15

In [75]:
# prepare peptides for viterbi calculation (expected to be unique)
from hmm_logic_methods import calculate_model_core_guesses
import os
import mhcgnomes
from scipy import stats

selected_peptides = []
for allele_name in ALLELES:
    print(allele_name)
    
    target_df = data_df[data_df['allele'].map(lambda x : x in allele_name)]
    allele_peptides = target_df.index.values
    total_df, selected_peptides_to_paths = calculate_model_core_guesses(allele_peptides, per_name_models, per_model_important_states, per_model_skip_states=per_model_skip_states,
                                    selected_peptides=selected_peptides
                                   )
    
    # set core as most popular
    models_predictions_start = total_df[[f"core_start_{i}" for i in range(RUNS)]]
    models_predictions_end = total_df[[f"core_end_{i}" for i in range(RUNS)]]
    model_predictions_cores = total_df[[f"core_{i}" for i in range(RUNS)]]
    core_consensuses = model_predictions_cores.apply(lambda x: 
                    stats.mode(np.vstack(x.values))[0][0], axis=1)
    core_starts = models_predictions_start.mode(axis=1)[0].astype(int)
    core_ends = models_predictions_end.mode(axis=1)[0].astype(int)
    
    final_df = total_df.copy()
    selected_cluster_column = 'min_entropy_cluster'
    #selected_cluster_column = 'max_mean_prob_cluster'
    # final_df['core_start'] = 0
    # final_df['core_end'] = 0
    # for cluster in cluster_descriptions.index:
    #     index_peptides = final_df[final_df[selected_cluster_column] == cluster].index
    #     start_values = cluster_to_start_columns[cluster]
    #     end_values = cluster_to_end_columns[cluster]
    #     final_df.loc[index_peptides, 'core_start'] = start_values.loc[index_peptides].mean(axis=1).round().astype(int).values
    #     final_df.loc[index_peptides, 'core_start'] = start_values.loc[index_peptides].mode(axis=1)[0].astype(int)

    #     #final_df.loc[index_peptides, 'core_end'] = end_values.loc[index_peptides].mean(axis=1).round().astype(int).values
    #     final_df.loc[index_peptides, 'core_end'] = end_values.loc[index_peptides].mode(axis=1)[0].astype(int)

    final_df['core_start'] = core_starts
    final_df['core_end'] = core_ends

    #final_df['core_peptide'] = final_df.apply(lambda row: row.name[row['core_start']:row['core_end'] + 1], axis=1)
    final_df['core_peptide'] = final_df.apply(lambda row: "".join(np.array(list(row.name))[core_consensuses.loc[row.name].astype(bool)]), axis=1)

    final_df['core_len'] = final_df['core_peptide'].str.len()
    
    print(final_df['core_len'].value_counts())
    
    target_df['core_start'] = final_df['core_start']
    target_df['core_end'] = final_df['core_end']
    target_df['core_len'] = final_df['core_len']
    target_df['peptide_len'] = target_df.index.str.len()
    target_df['core_peptide'] = final_df['core_peptide']
    
    target_df = target_df[~target_df.index.duplicated(keep='first')]
    target_df['peptide']= target_df.index.values
    
    parsed_alleles = [mhcgnomes.parse(curr_name).compact_string(include_species=True).replace("*", "_") for curr_name in allele_name.split("+")]
    target_allele_name = "+".join(parsed_alleles)
    
    #arget_allele_name = mhcgnomes.parse(allele_name).compact_string(include_species=True).replace("*", "_")
    if not os.path.exists(PATH_TO_SAVE_RESULTS):
        os.makedirs(PATH_TO_SAVE_RESULTS)
    target_df.to_csv("{}cores#{}.csv".format(PATH_TO_SAVE_RESULTS,target_allele_name),sep=";", index=True)
    target_df.peptide.to_csv("{}peptides#{}.csv".format(PATH_TO_SAVE_RESULTS,target_allele_name),sep=";", index=False)






len(allele_peptides)

HLA-DPA1*02:02/DPB1*05:01
Model 0_run-12-13-14-15-16-17-18-14-st-viterbi-alg-1000-min_iters_model-HLA_DPA1_02_02_DPB1_05_01-0
Get viterbi paths for each peptide...


100%|██████████| 5640/5640 [00:00<00:00, 21609.16it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 5640 


100%|██████████| 5640/5640 [00:00<00:00, 20853.72it/s]


Model 1_run-12-13-14-15-16-17-18-14-st-viterbi-alg-1000-min_iters_model-HLA_DPA1_02_02_DPB1_05_01-0
Get viterbi paths for each peptide...


100%|██████████| 5640/5640 [00:00<00:00, 25725.63it/s]


... (92 outputs left out)

Subselect unique paths (if unique path argument passed)..
	Unique paths: 5640 


100%|██████████| 5640/5640 [00:00<00:00, 18198.41it/s]
C:\Projects\MHC_predictor\experiments\core_identification_geometrical_cycle\hmm_logic_methods.py:1749: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df[f'prob_{i}'] =  viterbi_paths['norm_viterbi_prob']
C:\Projects\MHC_predictor\experiments\core_identification_geometrical_cycle\hmm_logic_methods.py:1754: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df['peptide'] = total_df.index.values
C:\Users\denkl\AppData\Local\Temp\ipykernel_9280\691033782.py:22: FutureWarni

9     4327
10    1207
11     103
12       3
Name: core_len, dtype: int64


5640

In [76]:
TEST_PDB_CORES = False
if TEST_PDB_CORES:
    df=pd.read_csv('test/binding_cores', sep='\t')
    df= df[~df['Antigen'].str.contains('X')]
    df = df[df['Allele'] == 'DRB1*01:01']

    allele_peptides = df.Antigen.values
    true_labels=df['Core (PDB)'].values
    netmhcpan = df['Predicted core.5'].values

In [77]:
from hmm_logic_methods import calculate_model_core_guesses
selected_peptides = [
    # "APEEILAEKSFNTVPY", 'EALTFSREHGPGVAEPE',
    #                  'YYSLLMPILTLTRAL',
    #                      'SYLTNMFATWSP',
    #                  'NPLIRHENRMVLAST',
    #                      'KPPKPVSKMRMATPLLM',
                         ]
total_df, selected_peptides_to_paths = calculate_model_core_guesses(allele_peptides, per_name_models, per_model_important_states, per_model_skip_states=per_model_skip_states,
                                    selected_peptides=selected_peptides
                                   )

Model 0_run-12-13-14-15-16-17-18-14-st-viterbi-alg-1000-min_iters_model-HLA_DPA1_02_02_DPB1_05_01-0
Get viterbi paths for each peptide...


100%|██████████| 5640/5640 [00:00<00:00, 29663.80it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 5640 


100%|██████████| 5640/5640 [00:00<00:00, 24742.02it/s]


Model 1_run-12-13-14-15-16-17-18-14-st-viterbi-alg-1000-min_iters_model-HLA_DPA1_02_02_DPB1_05_01-0
Get viterbi paths for each peptide...


100%|██████████| 5640/5640 [00:00<00:00, 30990.49it/s]


... (90 outputs left out)

Model 24_run-12-13-14-15-16-17-18-14-st-viterbi-alg-1000-min_iters_model-HLA_DPA1_02_02_DPB1_05_01-0
Get viterbi paths for each peptide...


100%|██████████| 5640/5640 [00:00<00:00, 22818.04it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 5640 


100%|██████████| 5640/5640 [00:00<00:00, 19952.91it/s]
C:\Projects\MHC_predictor\experiments\core_identification_geometrical_cycle\hmm_logic_methods.py:1749: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df[f'prob_{i}'] =  viterbi_paths['norm_viterbi_prob']
C:\Projects\MHC_predictor\experiments\core_identification_geometrical_cycle\hmm_logic_methods.py:1754: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df['peptide'] = total_df.index.values


In [78]:
data_df = data_df.join(total_df)


In [79]:
if TEST_PDB_CORES:
    models_predictions_start = total_df[[f"core_start_{i}" for i in range(RUNS)]]
    models_predictions_end = total_df[[f"core_end_{i}" for i in range(RUNS)]]
    core_starts = models_predictions_start.mode(axis=1)[0].astype(int)
    core_ends = models_predictions_end.mode(axis=1)[0].astype(int)
    
    total_df['core_start'] = core_starts
    total_df['core_end'] = core_ends

    total_df['core_peptide'] = total_df.apply(lambda row: row.name[row['core_start']:row['core_end'] + 1], axis=1)
    total_df['core_len'] = total_df['core_peptide'].str.len()
    total_df['peptide_len'] = total_df.index.str.len()
    total_df['core_peptide'] = total_df['core_peptide']
    total_df = total_df[~total_df.index.duplicated(keep='first')]


    total_df.to_csv('test/predicted_cores_cores.csv',sep=";", index=True)

## Set core based on most popular value

In [80]:
from scipy import stats

models_predictions_start = total_df[[f"core_start_{i}" for i in range(RUNS)]]
models_predictions_end = total_df[[f"core_end_{i}" for i in range(RUNS)]]
model_predictions_cores = total_df[[f"core_{i}" for i in range(RUNS)]]
core_consensuses = model_predictions_cores.apply(lambda x: 
                stats.mode(np.vstack(x.values))[0][0], axis=1)


C:\Users\denkl\AppData\Local\Temp\ipykernel_9280\2287683951.py:7: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  stats.mode(np.vstack(x.values))[0][0], axis=1)


In [81]:

core_consensuses

MPQVPVFTVEVKNYGVK     [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, ...
RTELETLQKQKK          [0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...
SQEKQVTVLELFR         [0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...
IIMDVYSYSEKE          [0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...
QALVEHVKEECD          [0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...
                                            ...                        
GATGAIMTTETTKTARAT    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, ...
STKTSAKLSSYT          [0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...
IASRLGLYSKTSQS        [0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...
ELSAWTEKQKEERQF       [0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...
SVEEKRNSKMLQK         [0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...
Length: 5640, dtype: object

In [82]:
len([item for item in core_consensuses if 0 in item])

5640

In [83]:
core_starts = models_predictions_start.mode(axis=1)[0].astype(int)
core_ends = models_predictions_end.mode(axis=1)[0].astype(int)


In [84]:
final_df = total_df

In [85]:
selected_cluster_column = 'min_entropy_cluster'
#selected_cluster_column = 'max_mean_prob_cluster'
# final_df['core_start'] = 0
# final_df['core_end'] = 0
# for cluster in cluster_descriptions.index:
#     index_peptides = final_df[final_df[selected_cluster_column] == cluster].index
#     start_values = cluster_to_start_columns[cluster]
#     end_values = cluster_to_end_columns[cluster]
#     final_df.loc[index_peptides, 'core_start'] = start_values.loc[index_peptides].mean(axis=1).round().astype(int).values
#     final_df.loc[index_peptides, 'core_start'] = start_values.loc[index_peptides].mode(axis=1)[0].astype(int)

#     #final_df.loc[index_peptides, 'core_end'] = end_values.loc[index_peptides].mean(axis=1).round().astype(int).values
#     final_df.loc[index_peptides, 'core_end'] = end_values.loc[index_peptides].mode(axis=1)[0].astype(int)

final_df['core_start'] = core_starts
final_df['core_end'] = core_ends

#final_df['core_peptide'] = final_df.apply(lambda row: row.name[row['core_start']:row['core_end'] + 1], axis=1)
final_df['core_peptide'] = final_df.apply(lambda row: "".join(np.array(list(row.name))[core_consensuses.loc[row.name].astype(bool)]), axis=1)

final_df['core_len'] = final_df['core_peptide'].str.len()

In [86]:
final_df['core_len'].value_counts()

9     4327
10    1207
11     103
12       3
Name: core_len, dtype: int64

In [87]:
data_df['core_start'] = final_df['core_start']
data_df['core_end'] = final_df['core_end']
data_df['core_len'] = final_df['core_len']
data_df['peptide_len'] = data_df.index.str.len()
data_df['core_peptide'] = final_df['core_peptide']
data_df = data_df[~data_df.index.duplicated(keep='first')]

C:\Users\denkl\AppData\Local\Temp\ipykernel_9280\1549749149.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_df['core_start'] = final_df['core_start']
C:\Users\denkl\AppData\Local\Temp\ipykernel_9280\1549749149.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_df['core_end'] = final_df['core_end']
C:\Users\denkl\AppData\Local\Temp\ipykernel_9280\1549749149.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Conside

In [88]:
data_df['core_start'].value_counts()

4    1624
5    1459
6     826
3     818
2     426
7     286
8      84
1      71
0      31
9      15
Name: core_start, dtype: int64

# Save core data to files

In [89]:
import os
if not os.path.exists(PATH_TO_SAVE_RESULTS):
    os.makedirs(PATH_TO_SAVE_RESULTS)
data_df.to_csv("{}extractd_cores.csv".format(PATH_TO_SAVE_RESULTS,),sep=";", index=True)
data_df.peptide.to_csv("{}original_peptides.csv".format(PATH_TO_SAVE_RESULTS,),sep=";", index=False)

# Group models based on predictions

We got all the start predictions from all models. Now we want to group these models. (since we expect that models have disagrement)

We want to cluster columns based on how similar are predictions for the whole training data



In [65]:
models_predictions = total_df[[f"core_start_{i}" for i in range(RUNS)]]

In [ ]:
from sklearn.cluster import FeatureAgglomeration
distance_threshold = 150

agglo = FeatureAgglomeration(n_clusters=None, metric='euclidean', linkage='ward', pooling_func=np.mean, distance_threshold=distance_threshold)
result_start_df = agglo.fit_transform(models_predictions)

In [78]:
clustered_features = pd.DataFrame({"cluster": agglo.labels_})

clustered_features['model'] = models_predictions.columns


unique_labels_counts = clustered_features.cluster.value_counts().sort_values(ascending=False)
unique_labels_counts

1    15
2    10
3     6
0     4
Name: cluster, dtype: int64

In [79]:
cluster_descriptions = clustered_features.groupby('cluster').agg(lambda x: " ".join(x.values))
cluster_descriptions['len'] = unique_labels_counts
cluster_descriptions.sort_values(by='len', ascending=False)

,model,len
cluster,,
1,core_start_1 core_start_3 core_start_10 core_s...,15
2,core_start_2 core_start_5 core_start_6 core_st...,10
3,core_start_9 core_start_11 core_start_23 core_...,6
0,core_start_0 core_start_4 core_start_17 core_s...,4


In [80]:
cluster_to_start_columns = {i : total_df[cluster_descriptions.loc[i]['model'].split(' ')] for i in cluster_descriptions.index}

In [81]:
model_to_cluster = dict(clustered_features['cluster'])

## Add mean probability for cluster for each peptide

In [34]:
from collections import defaultdict
cluster_to_prob_column_names = defaultdict(list)
cluster_to_end_column_names = defaultdict(list)

for cluster in cluster_descriptions.index:
    print(cluster)
    total_df[f'cluster_prob_{cluster}'] = 0
    
# calculate mean for predictions for given cluster of models to 
for i in range(RUNS):
    total_df[f'cluster_prob_{model_to_cluster[i]}'] += total_df[f'prob_{i}']
    cluster_to_prob_column_names[model_to_cluster[i]].append(f'prob_{i}')
    cluster_to_end_column_names[model_to_cluster[i]].append(f'core_end_{i}')

for cluster in cluster_descriptions.index:
    print(cluster)
    total_df[f'cluster_prob_{cluster}'] = total_df[f'cluster_prob_{cluster}'] / cluster_descriptions.loc[cluster]['len']
    
total_df['max_mean_prob_cluster'] =   total_df[[f'cluster_prob_{cluster}' for cluster in cluster_descriptions.index]].idxmax(axis=1).str.split('_').map(lambda x : int(x[-1]))

cluster_to_prob_columns = {i : total_df[cluster_to_prob_column_names[i]] for i in cluster_to_prob_column_names}
cluster_to_end_columns = {i : total_df[cluster_to_end_column_names[i]] for i in cluster_to_end_column_names}


0
1
2
3
0
1
2
3


In [35]:
cluster_to_start_columns[1].shape

(5640, 15)

## Add entropy value for cluster for each column

In [36]:
def mean_predictive_entropy(predictions):
    entropy = np.mean(-np.sum(np.exp(predictions)*predictions, axis=1)/predictions.shape[1])
    return entropy

def predictive_entropy(predictions):
    entropy = -np.sum(np.exp(predictions)*predictions, axis=1)/predictions.shape[1]
    return entropy

print(mean_predictive_entropy(cluster_to_prob_columns[0]))
print(mean_predictive_entropy(cluster_to_prob_columns[1]))
#print(mean_predictive_entropy(cluster_to_prob_columns[2]))
#print(mean_predictive_entropy(cluster_to_prob_columns[3]))

cluster_to_start_entropy_df = {i: predictive_entropy(cluster_to_prob_columns[i]) for i in cluster_to_prob_columns.keys()}

0.15289490292766392
0.15267749732179514


In [37]:
df_concat = pd.concat(cluster_to_start_entropy_df.values(), axis=1)
min_entropy_clusters = df_concat.idxmin(axis=1)
df_concat.columns = [f'cluster_entropy_{cluster}' for cluster in df_concat.columns ]
df_concat['min_entropy_cluster'] = min_entropy_clusters

In [38]:
final_df = pd.concat([total_df, df_concat], axis=1)